In [4]:
import os
import datetime
from langsmith import Client
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [5]:
kanana1 = "Kanana-test-0"
kanana2 = "Kanana-test-2"
kanana3 = "Kanana-test-3"

skt1 = "A.X-3.1-Light-test-0"
skt2 = "A.X-3.1-Light-test-1"
skt3 = "A.X-3.1-Light-test-3"

In [6]:
client = Client()

In [35]:
import json
from typing import Optional, Dict, Any
 
from langsmith import Client
 
 
def build_trace_dict(
    project_name: str,
    limit: Optional[int] = None,
) -> Dict[str, Dict[str, Any]]:
    """
    LangSmith 프로젝트 내의 모든 트레이스를 조회하여 dictionary로 변환합니다.
 
    동작 원리:
    1) is_root=True 로 각 트레이스의 최상위 run(예: 이미지의 'LangGraph')을 조회.
       -> 이 run의 id가 곧 '트레이스 id' 역할을 하며, inputs를 최상위 input으로 사용.
    2) 같은 트레이스에 속한 모든 child run을 trace_id로 다시 조회.
       -> 각 child run의 name이 노드 이름, outputs가 해당 노드의 output,
          end_time - start_time이 latency가 됩니다.
 
    Args:
        project_name: LangSmith 프로젝트 이름 (예: "A.X-3.1-Light-test-3")
        limit: 가져올 최대 트레이스(루트 run) 개수. None이면 전체 조회.
 
    Returns:
        run_id를 key로 하는 dictionary
    """
    client = Client()
    result: Dict[str, Dict[str, Any]] = {}
    print("project_name : " + project_name)
    # 1. 프로젝트 내 루트 run(=각 트레이스의 시작점) 조회
    root_runs = client.list_runs(
        project_name=project_name,
        is_root=True,
        limit=limit,
    )
 
    for root_run in root_runs:
        run_id = str(root_run.id)
        print(run_id)
        trace_id = root_run.trace_id
 
        entry: Dict[str, Any] = {
            "input": root_run.inputs,
        }
 
        # 2. 동일 트레이스(trace_id)에 속한 모든 child run 조회
        child_runs = client.list_runs(
            project_name=project_name,
            trace_id=trace_id,
        )
 
        for run in child_runs:
            if run.id == root_run.id:
                # 루트 run 자체는 위에서 input으로 이미 처리했으므로 건너뜀
                continue
 
            latency = None
            if run.start_time and run.end_time:
                latency = (run.end_time - run.start_time).total_seconds()
 
            node_name = run.name
            # 동일 이름의 노드가 한 트레이스 안에서 여러 번 실행된 경우
            # 덮어써지지 않도록 이름 뒤에 숫자를 붙여 구분
            if node_name in entry:
                suffix = 2
                candidate = f"{node_name}_{suffix}"
                while candidate in entry:
                    suffix += 1
                    candidate = f"{node_name}_{suffix}"
                node_name = candidate
 
            entry[node_name] = {
                "output": run.outputs,
                "latency": latency,
            }
 
        result[run_id] = entry
 
    return result

In [36]:
k1 = build_trace_dict(kanana1)
k2 = build_trace_dict(kanana2)
k3 = build_trace_dict(kanana3)

s1 = build_trace_dict(skt1)
s2 = build_trace_dict(skt2)
s3 = build_trace_dict(skt3)

project_name : Kanana-test-0
019ecfbb-0f5d-7f10-b51a-4e30c0feb592
019ecfba-d640-7192-84ab-0032dd9458b3
019ecfba-9eec-7283-9e8d-0ebd0513bfb0
019ecfab-7f56-7750-828e-da4a5768e170
019ecfab-3496-7213-9eae-c72292e5b832
019ecfaa-fb5a-7e41-a2b9-113fad5d1bdc
019ecfa2-531a-7203-aa28-8336bde5ee78
019ecfa1-fc76-70b0-a0b4-f351e622ee0f
019ecfa1-ae85-70c3-a6ac-e212654f9ac7
019ecfa1-6041-7ba3-bd10-6903a7c709d8
019ecfa1-2b70-7431-8cb2-80404de12519
019ecfa0-f2d8-7413-be0c-a42c6735e6ee
019ecf95-80a9-7100-b366-261ad86f0308
019ecf8a-3c6e-7550-b7e6-0c992691901e
019ecf89-e1df-78e1-8e49-a2a05ba0aab8
019ecf89-a988-70d0-abb4-15579b49d1b9
019ecf89-2955-7433-8a24-075dbd099bc6
019ecf88-f54c-7461-a34b-a40757237429
019ecf88-c0a0-7663-880f-33983d47df53
019ecf88-6d9c-7652-a8df-c6d01cf5cb49
019ecf88-1339-7062-9c36-cae6c6d7f3ed
019ecf87-db19-7a00-88df-05a59a34c252
019ecf87-8ef7-7b91-9918-09c7c3446929
019ecf87-4832-7641-8b3c-7ecc27d2593d
019ecf87-0a22-7442-b5b0-b1513162d8d6
019ecf86-be8a-7fd3-a464-c093d153453f
019ecf86-

In [51]:
s2[list(s2.keys())[3]]

{'input': {'raw_incident_data': {'endDateTime': '2026-06-12 18:00',
   'gu': '성동구',
   'info': '올림픽대로 김포방향(동호대교→한남대교) 4차로 추돌사고',
   'lat': '127.02650391450922',
   'lng': '37.53380779156013',
   'si': '서울특별시',
   'startDateTime': '2026-06-12 16:20:00',
   'x': '202342.563393',
   'y': '448256.690943'}},
 'node_apply_neo4j': {'output': {'final_outputs': [{'affected': '올림픽대로 김포방향-동호대교-한남대교',
     'content': '올림픽대로 김포방향(동호대교→한남대교) 4차로 추돌사고로 인한 통제',
     'details': {'end_node': '한남대교',
      'road_name': '올림픽대로 김포방향',
      'start_node': '동호대교'},
     'endDateTime': '2026-06-12 18:00:00',
     'gu': '성동구',
     'incident_id': 'f647e343cd2fe6b8db21604a9ac4afd1',
     'lat': 37.53380779156013,
     'lng': 127.02650391450922,
     'location_type': 'BETWEEN_NODES',
     'obj': '통제',
     'si': '서울특별시',
     'startDateTime': '2026-06-12 16:20:00'}]},
  'latency': 0.050883},
 'node_enrich_coordinates': {'output': {'final_outputs': [{'affected': '올림픽대로 김포방향-동호대교-한남대교',
     'content': '올림픽대로 김포방향

In [54]:
len(s2.keys())

32

In [41]:
def save_to_json(data: Dict[str, Any], filepath: str) -> None:
    """결과를 JSON 파일로 저장합니다. (datetime 등 직렬화 불가능한 값은 문자열로 변환)"""
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2, default=str)
 
 
def load_from_json(filepath: str) -> Dict[str, Any]:
    """save_to_json으로 저장한 JSON 파일을 다시 dictionary로 불러옵니다."""
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)

In [52]:
save_to_json(k1, 'kanana1.json')
save_to_json(k2, 'kanana2.json')
save_to_json(k3, 'kanana3.json')

save_to_json(s1, 'skt1.json')
save_to_json(s2, 'skt2.json')
save_to_json(s3, 'skt3.json')